# EV fleet dispatch – ablation runs (Ocean Park/Hanoi simulator)

Runs part of `scripts/run_ablation.py` on Kaggle CPUs. The simulator is pure Python on CPU,
so a GPU does not help: choose **no accelerator**. Speed comes from running several copies of
this notebook at the same time, each with a different `SHARD`.

1. Add the dataset that holds `vric.zip` (Kaggle may unpack it), `versions.json`, `fingerprint.json`
   (from `kaggle/make_bundle.py`); set `DATASET` in the setup cell to its path.
2. Settings: Internet **on** (to pin library versions), accelerator **None**.
3. Set `SHARD` (0 … `NSHARDS`-1), then *Save Version → Save & Run All*.
4. When it finishes, download `results_<stage>_shard<k>.zip` from the Output tab and run
   `scripts/merge_kaggle_results.py` on your machine.

In [ ]:
STAGE = "full"   # "full" = 5,080 cars (groups A, E); "small" = 1,000 cars (groups A-D)
SHARD = 0        # this copy's part: 0 .. NSHARDS-1
NSHARDS = 5      # number of copies running in parallel (full stage has 5 configs to run)

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile

DATASET = "/kaggle/input/datasets/minh12605/evsim-vric"   # where the bundle dataset is mounted
WORK = "/kaggle/working/vric"
# Kaggle unpacks uploaded .zip files, so the code may sit in a folder instead of vric.zip
BUNDLE = os.path.dirname(glob.glob(f"{DATASET}/**/versions.json", recursive=True)[0])
zips = glob.glob(f"{DATASET}/**/vric.zip", recursive=True)
if zips:
    os.makedirs(WORK, exist_ok=True)
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    code_root = os.path.dirname(glob.glob(f"{DATASET}/**/run_baseline.py", recursive=True)[0])
    shutil.copytree(code_root, WORK, dirs_exist_ok=True)   # the input folder is read-only
for need in ["run_baseline.py", "evsim/simulator.py", "scripts/run_ablation.py", "data/road_cells.npz",
             "data/stations_google.csv", "data/didi_values_hanoi_full.npz", "data/reject_curve.json"]:
    assert os.path.exists(f"{WORK}/{need}"), f"missing {need} in the dataset"
ver = json.load(open(f"{BUNDLE}/versions.json"))
print("bundle", BUNDLE, "| local versions", ver, "| Kaggle python", sys.version.split()[0], "| CPUs", os.cpu_count())

# same numpy/scipy/pandas as the machine that made the bundle, so seeds give the same riders
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"numpy=={ver['numpy']}",
                f"scipy=={ver['scipy']}", f"pandas=={ver['pandas']}", "tabulate", "matplotlib"], check=True)

In [ ]:
# reproducibility check: a tiny run must match the one made on the local machine
check = r'''
import dataclasses, json, sys
sys.path.insert(0, ".")
from evsim.battery import POLICIES
from evsim.config import SCENARIOS, Config
from evsim.metrics import summarize
from evsim.simulator import Simulator
ref = json.load(open(sys.argv[1]))
out = {}
for policy, want in ref["metrics"].items():
    cfg = dataclasses.replace(Config(), seed=ref["seed"], **{**SCENARIOS[ref["scenario"]], **ref["overrides"]})
    got = summarize(Simulator(cfg, POLICIES[policy](cfg)).run())[0]
    out[policy] = {k: round(float(got[k]), 6) for k in want}
print(json.dumps(out))
'''
res = subprocess.run([sys.executable, "-c", check, f"{BUNDLE}/fingerprint.json"], cwd=WORK,
                     capture_output=True, text=True)
print(res.stderr[-2000:])
got = json.loads(res.stdout.strip().splitlines()[-1])
want = json.load(open(f"{BUNDLE}/fingerprint.json"))["metrics"]
REUSE_OK = got == want
print("fingerprint matches the local machine:", REUSE_OK)
if not REUSE_OK:
    print("random streams differ -> the black boxes will be run here too (--no-reuse)")
    print("local:", want); print("kaggle:", got)

In [ ]:
cmd = [sys.executable, "-u", "scripts/run_ablation.py", "--stage", STAGE, "--shard", str(SHARD),
       "--nshards", str(NSHARDS), "--jobs", str(os.cpu_count()), "--run-only"] + ([] if REUSE_OK else ["--no-reuse"])
print(" ".join(cmd))
p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
assert p.wait() == 0, "ablation run failed"


In [ ]:
# package this shard's results for download
out = f"/kaggle/working/results_{STAGE}_shard{SHARD}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root in glob.glob(f"{WORK}/outputs/hanoi_full_abl_*") + [f"{WORK}/outputs/runs"]:
        for f in glob.glob(f"{root}/**/*", recursive=True):
            if os.path.isfile(f) and "requests_seed0" not in f and "soc_snapshots" not in f:
                z.write(f, os.path.relpath(f, WORK))
    z.writestr("kaggle_meta.json", json.dumps({"stage": STAGE, "shard": SHARD, "nshards": NSHARDS,
                                               "reuse_ok": REUSE_OK, "python": sys.version.split()[0]}))
print("saved", out, round(os.path.getsize(out) / 1e6, 1), "MB")